# 3. Evaluation

Pooled out-of-fold results with bootstrap confidence intervals, McNemar tests between conditions,
tests across GA configurations, comparison with the baselines, and subset stability across seeds.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import binomtest, friedmanchisquare, wilcoxon
from sklearn.metrics import f1_score, recall_score

DATASET = "BUSI"  # or "BUS-UCLM"
OUT_DIR = Path("../outputs") / DATASET

data = joblib.load(OUT_DIR / "folds.pkl")
y, folds = data["y"], data["folds"]
runs = pd.DataFrame(joblib.load(OUT_DIR / "ga_runs.pkl"))
baselines = pd.DataFrame(joblib.load(OUT_DIR / "baselines.pkl"))
runs["f1"] = [f1_score(y[folds[k][1]], p, average="macro") for k, p in zip(runs.fold, runs.pred)]
runs["n_features"] = runs["mask"].apply(np.sum)

## Pooled out-of-fold results

Standard GA, seed 0. Test predictions of the five folds are pooled before scoring.

In [ ]:
def pooled(condition):
    part = runs.query("config == 'Standard' and seed == 0 and condition == @condition").sort_values("fold")
    truth = np.concatenate([y[folds[k][1]] for k in part.fold])
    return truth, np.concatenate(part.pred.tolist())


rng = np.random.default_rng(0)
rows = []
for condition in ["raw", "predicted", "shuffled", "oracle"]:
    truth, pred = pooled(condition)
    boot = [f1_score(truth[i], pred[i], average="macro")
            for i in (rng.integers(0, len(truth), len(truth)) for _ in range(1000))]
    recall = recall_score(truth, pred, average=None, labels=[0, 1, 2])
    rows.append(dict(dataset=DATASET, condition=condition,
                     macro_f1=f1_score(truth, pred, average="macro"),
                     ci_low=np.percentile(boot, 2.5), ci_high=np.percentile(boot, 97.5),
                     recall_benign=recall[0], recall_malignant=recall[1], recall_normal=recall[2]))
pooled_table = pd.DataFrame(rows).round(3)
pooled_table

In [ ]:
rows = []
for a, b in [("predicted", "raw"), ("oracle", "shuffled"), ("oracle", "raw"), ("shuffled", "raw")]:
    (truth, pa), (_, pb) = pooled(a), pooled(b)
    only_a = int(((pa == truth) & (pb != truth)).sum())
    only_b = int(((pa != truth) & (pb == truth)).sum())
    rows.append(dict(dataset=DATASET, a=a, b=b, a_only_correct=only_a, b_only_correct=only_b,
                     p_exact=binomtest(only_a, only_a + only_b).pvalue))
mcnemar_table = pd.DataFrame(rows).round(4)
mcnemar_table

## GA configurations

In [ ]:
summary = (runs.groupby(["condition", "config"])
               .agg(f1_mean=("f1", "mean"), f1_sd=("f1", "std"),
                    features=("n_features", "mean"), last_improvement=("last_improvement", "mean"))
               .round(3))
summary

In [ ]:
def holm(p):
    p = np.asarray(p)
    adjusted, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, (len(p) - rank) * p[i])
        adjusted[i] = min(1.0, running)
    return adjusted


def paired_p(a, b):
    return 1.0 if np.allclose(a, b) else wilcoxon(a, b).pvalue


rows = []
for condition, part in runs[runs.condition != "shuffled"].groupby("condition"):
    by_fold = part.pivot_table(index="fold", columns="config", values="f1")
    by_run = part.pivot_table(index=["fold", "seed"], columns="config", values="f1")
    others = [c for c in by_run if c != "Standard"]
    p_holm = holm([paired_p(by_run[c], by_run["Standard"]) for c in others])
    rows.append(dict(condition=condition,
                     friedman_p_folds=friedmanchisquare(*by_fold.T.values).pvalue,
                     friedman_p_runs=friedmanchisquare(*by_run.T.values).pvalue if len(by_run) > len(by_fold) else np.nan,
                     min_p_holm=p_holm.min(), config=others[int(np.argmin(p_holm))]))
pd.DataFrame(rows).round(3)

## Standard GA versus baselines

In [ ]:
methods = ["all_512", "anova", "mutual_info", "random"]
rows = []
for condition, part in baselines.groupby("condition"):
    for m in methods:
        diff = part.standard_ga - part[m]
        rows.append(dict(dataset=DATASET, condition=condition, k=part.k.mean().round(), baseline=m,
                         ga_mean=part.standard_ga.mean(), baseline_mean=part[m].mean(),
                         ga_wins=int((diff > 0).sum()), p_wilcoxon=paired_p(part.standard_ga, part[m])))
baseline_table = pd.DataFrame(rows).round(3)
baseline_table

## Subset stability

Jaccard similarity between the subsets selected with seeds 0 and 1 in the same fold,
next to the value expected for two random subsets of the same sizes.

In [ ]:
rows = []
if runs.seed.nunique() > 1:
    for (condition, config, k), part in runs.groupby(["condition", "config", "fold"]):
        if len(part) < 2:
            continue
        a, b = (m.astype(bool) for m in part.sort_values("seed")["mask"])
        expected = a.sum() * b.sum() / len(a)
        rows.append(dict(condition=condition, config=config,
                         jaccard=(a & b).sum() / (a | b).sum(),
                         random=expected / (a.sum() + b.sum() - expected)))
stability = pd.DataFrame(rows).groupby(["condition", "config"]).mean().round(3) if rows else pd.DataFrame()
stability

In [ ]:
for name, table in [("pooled", pooled_table), ("mcnemar", mcnemar_table), ("baselines", baseline_table)]:
    table.to_csv(OUT_DIR / f"{name}.csv", index=False)